In [ ]:
# 1.层和块，首先我们回顾一下MLP
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

X = torch.randn(2, 20)
net(X)

tensor([[-0.1248,  0.2377, -0.1208, -0.2214,  0.1000,  0.1955, -0.2048,  0.0171,
         -0.0280,  0.0163],
        [ 0.0363, -0.1327, -0.1117, -0.0719, -0.1983,  0.1530, -0.0361, -0.2103,
          0.0641,  0.0343]], grad_fn=<AddmmBackward0>)

In [ ]:
# nn.Sequential定义了一种特殊的Module，而nn.Module在pytorch 中尤为重要
class MLP(nn.Module):
    def __init__(self):
        super().__init__() # 调用父类的构造函数
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
        
    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))
net = MLP()
net(X)

tensor([[ 0.2277, -0.3914, -0.3876, -0.1434, -0.2034,  0.1194,  0.1148, -0.1920,
         -0.0006,  0.0318],
        [ 0.1013, -0.0980, -0.0988, -0.2103,  0.5154, -0.1334, -0.0461, -0.0719,
         -0.0849, -0.1172]], grad_fn=<AddmmBackward0>)

In [4]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for block in args:
            self._modules[block] = block
    
    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-0.4723,  0.1963,  0.2937, -0.1465,  0.2159,  0.2039,  0.4403,  0.0489,
          0.2048, -0.1827],
        [-0.3307,  0.3072, -0.0153, -0.1702,  0.1596, -0.0218,  0.4101,  0.2201,
          0.0527,  0.1620]], grad_fn=<AddmmBackward0>)

In [5]:
# 这样的框架提供了一种方便的方式来定义复杂的神经网络模型，而且也可以根据需要进行修改和扩展。
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()
net = FixedHiddenMLP()
net(X)

tensor(-0.0174, grad_fn=<SumBackward0>)

In [ ]:
# 除此之外我们还可以混合搭配各种组合。
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), 
            nn.ReLU(),
            nn.Linear(64, 32), 
            nn.ReLU())
        self.linear = nn.Linear(32, 16)
    
    def forward(self, X):
        return self.linear(self.net(X))
chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

In [ ]:
# 2.参数管理
# 在我们选择好了模型架构，并设置了合适的超参数后，我们就进入了训练阶段。训练完成后我们就得到了一个模型，有时候我们需要这个模型的参数来进行一些后续的操作，比如模型的部署。
# 此时就可以进行参数管理，依旧先用简单的MLP举例
net = nn.Sequential(
    nn.Linear(4, 8), # 4个输入，8个输出，对应的矩阵就是4列8行
    nn.ReLU(),
    nn.Linear(8, 1) # 8列1行
)
X = torch.rand(size=(2, 4))
net(X)

tensor([[0.1506],
        [0.1054]], grad_fn=<AddmmBackward0>)

In [11]:
# 训练好了可以通过索引来访问模型的任意层。这就像模型是一个列表一样，每层的参数都可以通过索引来访问。
# 例如，我们可以访问第一个全连接层的参数：
print(net[0].state_dict())
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.1154, -0.3340, -0.2095,  0.2181],
        [-0.3580,  0.3757,  0.1683,  0.0345],
        [-0.3414, -0.3935, -0.2064, -0.4579],
        [-0.1238, -0.0382, -0.2034,  0.4803],
        [-0.0441, -0.1396, -0.4552,  0.1265],
        [-0.3161,  0.4496, -0.2654,  0.4308],
        [ 0.4590, -0.2526, -0.3557,  0.2488],
        [-0.2818,  0.2235,  0.2299,  0.3467]])), ('bias', tensor([ 0.0022, -0.0897,  0.2737,  0.4366, -0.0410, -0.0166,  0.3849, -0.0464]))])
OrderedDict([('weight', tensor([[-0.0869,  0.2243,  0.2553, -0.3087, -0.2149,  0.2781, -0.0758, -0.1248]])), ('bias', tensor([0.2453]))])


In [13]:
# 也可以访问具体的参数
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)
# 也可以使用.grad访问梯度
print(net[2].weight.grad)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.2453], requires_grad=True)
tensor([0.2453])
None


In [15]:
# 也可以一次性访问所有参数
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])
# 或者使用下面的
print(net[2].state_dict())
print(net.state_dict()['2.bias'].data)

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))
OrderedDict([('weight', tensor([[-0.0869,  0.2243,  0.2553, -0.3087, -0.2149,  0.2781, -0.0758, -0.1248]])), ('bias', tensor([0.2453]))])
tensor([0.2453])


In [16]:
# 也可以从嵌套块收集参数
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.4105],
        [-0.4105]], grad_fn=<AddmmBackward0>)

In [17]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [18]:
# 因为是嵌套的，所以参数的访问也会嵌套
print(rgnet[0][1][0].bias.data)

tensor([-0.4982,  0.3250, -0.1320, -0.3996, -0.1645, -0.0321, -0.1064,  0.0484])


In [19]:
# 初始化参数
# 默认情况下，pytorch会根据一个范围均匀地初始化权重和偏置矩阵，这个范围是根据输入和输出的维度计算出来的。
# 例如，一个4列8行的矩阵，它的范围就是[-sqrt(4/8), sqrt(4/8)]

# 首先我们先调用内置的初始化器
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]


(tensor([ 0.0111, -0.0075, -0.0072,  0.0148]), tensor(0.))

In [20]:
# 我们也可以手动初始化参数
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [21]:
# 我们也可以对某些块应用不同的初始化方法，比如第一个神经网络层用Xavier，第三个初始化为常量42
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)


tensor([ 0.3794, -0.4170,  0.5724, -0.3254])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [22]:
# 有时候需要自己定义一种初始化方法，这里我们就可以自定义初始化
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0]) # 列表推导式+元组解包+切片
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[0.0000, 9.8137, -0.0000, -0.0000],
        [0.0000, 7.7947, 0.0000, -0.0000]], grad_fn=<SliceBackward0>)

In [23]:
# 我们也可以直接对参数进行操作
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000, 10.8137,  1.0000,  1.0000])

In [28]:
# 有时我们希望在多个层之间共享参数，我们可以定义一个稠密层，然后使用它的参数来设置另一个层的参数

shared = nn.Linear(8, 8)
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    shared,
    nn.ReLU(),
    shared,
    nn.ReLU(),
    nn.Linear(8, 1)
)
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们是同一个对象而不是说只是值相同
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


In [29]:
# 3.自定义层
# 有时候我们需要构造特定的层来执行特定的任务，这时候就需要自定义层
# 首先我们构造一个没有任何参数的层

class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [30]:
net = nn.Sequential(
    nn.Linear(8, 128), CenteredLayer(),
    )
Y = net(torch.rand(4, 8))
Y.mean()

tensor(-1.1642e-09, grad_fn=<MeanBackward0>)

In [31]:
# 定义一个带参数的层
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[-0.8460,  0.4727,  0.7660],
        [-0.3375,  0.7489, -1.2367],
        [-1.7718, -1.0045,  1.0535],
        [ 1.4763,  0.9717,  0.0473],
        [-0.9985, -1.0775,  0.7912]], requires_grad=True)

In [32]:
linear(torch.rand(2, 5))
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.],
        [0.]])

In [33]:
# 4.读写文件
# 有时候训练好的模型需要保存下来，或者在其他地方使用，这时候就需要读写文件
# 从最简单的张量开始

x = torch.arange(4)
torch.save(x, 'x-file')
x2 = torch.load("x-file")
x2

C:\Users\26650\AppData\Local\Temp\ipykernel_33728\4293267724.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x2 = torch.load("x-file")


tensor([0, 1, 2, 3])

In [ ]:
# 也可以存list
y = torch.zeros(4)
torch.save([x, y], 'x-files')
x2, y2 = torch.load('x-files')
(x2, y2)

In [ ]:
# 也可以存储一个字典
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

In [34]:
# 加载和保存模型参数
# 由于保存单层参数过于麻烦，这个pytorch框架提供可以加载和保存整个模型的方法

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)
    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
y = net(X)


In [36]:
torch.save(net.state_dict(), 'mlp.params')

clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

C:\Users\26650\AppData\Local\Temp\ipykernel_33728\765443990.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clone.load_state_dict(torch.load('mlp.params'))


MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

In [39]:
# 由于克隆模型有着相同的参数，所以输入相同的值会得到相同的输出
y_clone = clone(X)
y_clone == y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

'nvidia-Smi' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [42]:
# 5.GPU相关
# 查看GPU详情!nvidia-smi
torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [43]:
torch.cuda.device_count()

0

In [45]:
# 张量默认创建在CPU中，我们可以通过指定设备来将张量存储在GPU中
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
